
# 🏞️ Project 11 — Measure Lake Area

Excellent! This project is your **first quantitative remote sensing project**.

Until now, you've created **maps** that show where water exists.

Now you'll answer a scientific question:

> **How large is the lake?**

Instead of only seeing the water, you'll **measure its surface area**.

This is exactly the type of workflow used in:

- 💧 Reservoir monitoring
- 🌊 Drought assessment
- 🌾 Irrigation management
- 🌍 Climate change studies
- 🚨 Flood monitoring

---

# 🎯 Goal

Estimate the **surface area of a lake** from a Sentinel-2 satellite image.

Instead of simply producing a water map, you'll calculate **how many square meters (or square kilometers)** of water are present.

---

# 🆕 New Skill: Area Calculation

Until now, your workflow looked like this:

```text
Satellite Image
      ↓
Calculate NDWI
      ↓
Threshold
      ↓
Water Map
```

Now it becomes:

```text
Satellite Image
      ↓
Calculate NDWI
      ↓
Threshold
      ↓
Water Mask
      ↓
Measure Area
      ↓
Water Surface Area
```

Notice something important.

Until now your final result was an **image**.

Today your final result will be a **number**.

This is the beginning of **quantitative remote sensing**.

---

# 🤔 Before We Start

Imagine your water mask looks like this:

```text
1 1 1 0 0

1 1 0 0 0

1 1 1 1 0
```

where

```text
1 = Water
0 = Land
```

If every pixel represents

```text
10 m × 10 m
```

then one pixel covers

```text
100 square meters
```

If there are

```text
500 water pixels
```

then

```text
500 × 100

= 50,000 square meters
```

That is exactly what Earth Engine will help us calculate automatically.

---

# Step 1 — Choose a Study Area

We'll use the same study area from the previous projects.

```javascript
var cityArea = ee.Geometry.Polygon([
  [
    [51.55,32.55],
    [51.80,32.55],
    [51.80,32.75],
    [51.55,32.75],
    [51.55,32.55]
  ]
]);
```

Nothing new here.

We still need an area where Earth Engine knows where to search.

---

# Step 2 — Load the Sentinel-2 Collection

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);
```

Again, we load the Sentinel-2 image collection.

Notice that we are still working with an **ImageCollection**, not an image.

---

# Step 3 — Select One Image

```javascript
var image = collection
  .filterBounds(cityArea)
  .filterDate('2025-04-01','2025-04-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(cityArea);
```

This should feel familiar.

You've already used this workflow several times.

```text
Image Collection
        ↓
filterBounds()
        ↓
filterDate()
        ↓
sort()
        ↓
first()
        ↓
clip()
```

At the end,

```javascript
image
```

is an

```text
ee.Image
```

which we can now analyze.

---

# Step 4 — Calculate NDWI

Exactly like Project 9.

```javascript
var ndwi = image
  .normalizedDifference(['B3','B8'])
  .rename('NDWI');
```

Nothing new.

We're using:

```text
Green
and
NIR
```

to calculate water.

---

# Step 5 — Create a Water Mask

Now convert NDWI into a simple water/non-water image.

```javascript
var waterMask = ndwi.gt(0);
```

Remember from Project 8:

`.gt()` means

```text
Greater Than
```

Every pixel becomes

```text
1 = Water

0 = Not Water
```

So the image changes from

```text
NDWI

-0.12
0.44
0.61
-0.31
```

into

```text
0
1
1
0
```

Much easier to measure.

---

# ⭐ Step 6 — Create a Pixel Area Image

This is today's most important new concept.

```javascript
var pixelArea = ee.Image.pixelArea();
```

Let's understand this carefully.

Unlike previous images, this image **does not come from a satellite**.

Instead,

Earth Engine creates a brand-new image where **every pixel stores its own area**.

Think of it like this.

Satellite image:

```text
Red
Green
Blue
NIR
```

Pixel area image:

```text
100
100
100
100
```

For Sentinel-2,

each pixel is approximately

```text
10 × 10 meters
```

so each pixel stores approximately

```text
100 m²
```

This image is called

```text
pixelArea
```

and every pixel contains

its own area.

---

# ⭐ Step 7 — Keep Only Water Pixels

Now combine both images.

```javascript
var waterArea = pixelArea.updateMask(waterMask);
```

This line is extremely important.

Let's think visually.

Water mask:

```text
1 1 1 0 0

1 1 0 0 0
```

Pixel area image:

```text
100 100 100 100 100

100 100 100 100 100
```

After

```javascript
updateMask()
```

we get

```text
100 100 100

100 100
```

Land pixels disappear.

Only water pixels remain.

Now every remaining pixel stores

its own area.

---

# ⭐ Step 8 — Add All Water Pixels Together

Now comes another brand-new concept.

```javascript
var lakeArea = waterArea.reduceRegion({
  reducer: ee.Reducer.sum(),
  geometry: cityArea,
  scale: 10,
  maxPixels: 1e9
});
```

This is one of the most powerful functions in Earth Engine.

Let's understand the workflow first.

```text
Water Pixels

100
100
100
100
100

↓

Add Them

↓

500
```

Instead of returning another image,

Earth Engine calculates a summary value.

The function doing this is

```javascript
reduceRegion()
```

Think of it as

```text
Image

↓

Statistics

↓

Number
```

The reducer

```javascript
ee.Reducer.sum()
```

tells Earth Engine:

> "Add every remaining pixel together."

Since every pixel already stores its area,

adding them gives

the total lake area.

### Understanding the Parameters

### `reducer`

```javascript
reducer: ee.Reducer.sum()
```

Means:

> Add every pixel value.

---

### `geometry`

```javascript
geometry: cityArea
```

Means:

Only calculate inside

our study area.

---

### `scale`

```javascript
scale: 10
```

Means:

Use **10-meter pixels**.

Since Sentinel-2 has a 10 m spatial resolution for the bands we're using, this matches the data.

---

### `maxPixels`

```javascript
maxPixels: 1e9
```

Allows Earth Engine to process very large numbers of pixels without throwing an error.

You'll often see this parameter in large-area analyses.

---

# Step 9 — Print the Result

```javascript
print(
  'Lake Area (square meters)',
  lakeArea
);
```

Open the **Console**.

You'll see something like:

```text
Lake Area

5234687
```

This value is in

```text
square meters
```

---

# ⭐ Bonus Challenge — Convert to Square Kilometers

Scientists often report lake areas in

```text
km²
```

instead of

```text
m²
```

Remember:

```text
1 km²

=

1,000,000 m²
```

Convert the result.

```javascript
var lakeAreaKm2 = ee.Number(
  lakeArea.get('area')
).divide(1000000);

print(
  'Lake Area (km²)',
  lakeAreaKm2
);
```

Notice something new.

`reduceRegion()` returns a **Dictionary**, not a single number.

So we first use:

```javascript
lakeArea.get('area')
```

to retrieve the value stored under the band name (`area`, because `pixelArea()` creates a band with that name).

Then we convert it into an `ee.Number` so we can perform mathematical operations like `.divide()`.

---

# 💻 Complete Code

```javascript
// 1. Study Area
var cityArea = ee.Geometry.Polygon([
  [
    [51.55,32.55],
    [51.80,32.55],
    [51.80,32.75],
    [51.55,32.75],
    [51.55,32.55]
  ]
]);

// 2. Sentinel-2 Collection
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
);

// 3. Select Image
var image = collection
  .filterBounds(cityArea)
  .filterDate('2025-04-01','2025-04-30')
  .sort('CLOUDY_PIXEL_PERCENTAGE')
  .first()
  .clip(cityArea);

// 4. Calculate NDWI
var ndwi = image
  .normalizedDifference(['B3','B8'])
  .rename('NDWI');

// 5. Water Mask
var waterMask = ndwi.gt(0);

// 6. Pixel Area
var pixelArea = ee.Image.pixelArea();

// 7. Keep Only Water Pixels
var waterArea = pixelArea.updateMask(waterMask);

// 8. Calculate Total Area
var lakeArea = waterArea.reduceRegion({
  reducer: ee.Reducer.sum(),
  geometry: cityArea,
  scale:10,
  maxPixels:1e9
});

// 9. Print Result
print('Lake Area (square meters)', lakeArea);

// Bonus: Convert to km²
var lakeAreaKm2 = ee.Number(
  lakeArea.get('area')
).divide(1000000);

print('Lake Area (km²)', lakeAreaKm2);
```

---

# 🧠 A New Mental Model

This project introduced an entirely new workflow.

Previously, you created maps.

Now you extract quantitative information.

```text
Satellite Image
        ↓
Calculate NDWI
        ↓
Threshold
        ↓
Water Mask
        ↓
Pixel Area
        ↓
Sum All Water Pixels
        ↓
Lake Area (Number)
```

This is one of the core ideas of quantitative remote sensing.

---

# 📚 Things to Remember

## 1. This is your first quantitative analysis project

Previously:

```text
Image
   ↓
Map
```

Now:

```text
Image
   ↓
Number
```

Many remote sensing studies combine both:

- Maps for visualization
- Numbers for analysis and reporting

---

## 2. `ee.Image.pixelArea()`

Creates a new image where every pixel stores its own area in **square meters**.

It is **not** satellite data.

It is generated by Earth Engine.

---

## 3. `updateMask()`

You already learned this in Project 4 for cloud masking.

Today you used the exact same function in a different way.

Previously:

```text
Cloud Mask
      ↓
Hide Clouds
```

Today:

```text
Water Mask
      ↓
Keep Only Water
```

The function is the same.

Only the mask changed.

---

## 4. `reduceRegion()`

This is one of the most important Earth Engine functions.

It converts an **image** into one or more **summary values**.

Common reducers include:

| Reducer | Purpose |
|---------|----------|
| `sum()` | Total |
| `mean()` | Average |
| `min()` | Minimum |
| `max()` | Maximum |
| `median()` | Median |

You'll use `reduceRegion()` many times throughout your Earth Engine journey.

---

## 5. `ee.Reducer.sum()`

Adds together all the pixel values inside the specified region.

Since each remaining pixel represents its area in square meters, the sum gives the **total water surface area**.

---

## 6. `reduceRegion()` Returns a Dictionary

Unlike most functions you've used so far, `reduceRegion()` does **not** return an `ee.Image`.

It returns an `ee.Dictionary` that stores the calculated statistics.

That's why we use:

```javascript
lakeArea.get('area')
```

to retrieve the value before converting it into an `ee.Number`.

---

# ❌ Common Beginner Mistakes

### Mistake 1 — Measuring the NDWI Image Instead of the Water Mask

Incorrect:

```javascript
var waterArea = pixelArea.updateMask(ndwi);
```

`ndwi` contains continuous values between **-1 and 1**, not a binary mask.

Always create a water mask first.

```javascript
var waterMask = ndwi.gt(0);
```

---

### Mistake 2 — Forgetting to Apply the Mask

If you sum the entire `pixelArea` image,

you calculate the area of the **whole study area**, not just the lake.

Always keep only water pixels.

```javascript
pixelArea.updateMask(waterMask);
```

---

### Mistake 3 — Using the Wrong Scale

For Sentinel-2, use:

```javascript
scale:10
```

Choosing a scale that doesn't match your data can produce inaccurate area estimates or unnecessarily increase computation time.

---

### Mistake 4 — Forgetting That `reduceRegion()` Returns a Dictionary

This won't work:

```javascript
lakeArea.divide(1000000);
```

because `lakeArea` is a Dictionary, not a Number.

Retrieve the value first.

```javascript
ee.Number(
  lakeArea.get('area')
)
```

---

# 📝 Your Practice (Project 11.1)

Create your own lake area measurement project.

## Requirements

- Create a study area.
- Load the Sentinel-2 collection.
- Select one low-cloud image.
- Calculate NDWI.
- Create a water mask using `.gt()`.
- Create a `pixelArea` image.
- Keep only water pixels using `updateMask()`.
- Calculate the total lake area using `reduceRegion()`.
- Print the result in square meters.
- ⭐ Bonus: Convert the result to square kilometers.

---

## Before Writing the Code

Answer these two questions.

### Question 1

Why do we calculate the **water mask first** and then apply it to `pixelArea()`, instead of summing the entire `pixelArea` image directly?

---

### Question 2

`reduceRegion()` returns a **Dictionary** instead of a single number.

Why do you think Earth Engine was designed this way?

What kinds of calculations might require returning multiple values instead of just one?

---
